# Convert PDFs to structured Markdown with Docling

Turn layout-heavy PDFs into Markdown that you can chunk, embed, and search.

This notebook uses `pxt.create_table()` so you can run cells without a
project. An app puts the same columns on a `TableModel` in `app.py` and
creates the tables with `pxt schema update`.

## Problem

You have PDFs (reports, papers, mixed folders of files) that need to be
searchable. Raw PDF text is messy: columns, tables, and headers often
flatten in the wrong order. You want structured Markdown stored next to
the source files, and you do not want to reconvert a document unless it
changes.

| Source | What you need | What goes wrong |
|--------|---------------|-----------------|
| PDF report | Section-aware chunks | Columns concatenate in the wrong order |
| Research paper | Headings and tables | Page headers leak into every chunk |
| Folder of PDFs | Convert once, search often | Every query re-runs conversion |

## Solution

**What's in this recipe:**

- Convert PDFs with Docling as a computed column
- Store and version the Markdown, converting only new or changed files
- Chunk the Markdown and build an embedding index for retrieval

You wrap [Docling](https://github.com/docling-project/docling) in a
User-Defined Function (UDF) and add it as a computed column on a
document table. Pixeltable stores the Markdown, versions it with the
table, and converts only new or updated rows. You then split the
Markdown with `document_splitter` and add an embedding index for
similarity search.

You can iterate on the conversion before adding it to your table. Use
`.select()` with `.head(n)` to preview one document. Nothing is stored
until you call `.add_computed_column()`.

For more on this workflow, see
[Get fast feedback on transformations](../core/dev-iterative-workflow.ipynb).

### Setup

In [ ]:
%pip install -qU pixeltable docling sentence-transformers mistune tiktoken


### Load documents

In [2]:
import pixeltable as pxt
from pixeltable.functions.document import document_splitter
from pixeltable.functions.huggingface import sentence_transformer

# Create a fresh directory (drop existing if present)
pxt.drop_dir('docling_demo', force=True)
pxt.create_dir('docling_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:***@/pixeltable?host=%2Fhome%2Fubuntu%2F.pixeltable%2Fpgdata
Created directory 'docling_demo'.


In [3]:
docs = pxt.create_table(
    'docling_demo/documents',
    {'title': pxt.String, 'document': pxt.Document},
)

Created table 'documents'.


In [4]:
# Small public PDF already used in the RAG cookbooks
docs.insert(
    [
        {
            'title': 'Argus Market Digest',
            'document': (
                'https://raw.githubusercontent.com/'
                'pixeltable/pixeltable/main/docs/'
                'resources/rag-demo/'
                'Argus-Market-Digest-June-2024.pdf'
            ),
        }
    ]
)

Inserted 1 row with 0 errors in 0.06 s (15.44 rows/s)


1 row inserted.

### Define function to convert PDFs to Markdown

Defining a UDF makes the conversion easier to iterate on and easier to
reuse when you commit it as a computed column. Docling downloads layout
models on the first call. Later rows reuse the cached converter.

OCR is off because this sample is a digital PDF. Enable OCR in
`PdfPipelineOptions` when the source is a scan.

In [5]:
from functools import lru_cache


@lru_cache(maxsize=1)
def _docling_converter():
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import (
        DocumentConverter,
        PdfFormatOption,
    )

    options = PdfPipelineOptions()
    # Digital PDFs already contain text. Skip OCR.
    options.do_ocr = False
    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=options)
        }
    )


@pxt.udf
def to_markdown(doc: pxt.Document) -> str:
    result = _docling_converter().convert(doc)
    return result.document.export_to_markdown()

### Test conversion with a query

Use `.select()` with `.collect()` to preview results. If you want to
collect only the first few rows, use `.head(n)` instead of
`.collect()`.

In [6]:
# Preview Markdown for one PDF. Nothing is stored yet.
docs.select(docs.title, markdown=to_markdown(docs.document)).head(1)

title,markdown
Argus Market Digest,"<!-- image --> <!-- image --> <!-- image --> FRIDAY, JUNE 21, 2024 JUNE 20, DJIA: 39,134.76 UP 299.90 Good Morning. This is the Market Digest for Friday, June 21, 2024, with analysis of the financial markets and comments on Accenture plc. ## IN THIS ISSUE: * Growth Stock: Accenture plc: Shares rally on AI optimism (Jim Kelleher) ## MARKET REVIEW: Yogi Berra famously said ""When you come to a fork in the road, take it."" Stock investors did just that on Thursday, pushing the Dow Jones I ...... ures Act, the Intermediary accepts legal responsibility for the contents of Research Reports in respect of such recipient in accordance with applicable law. When reports are distributed by Intermediaries in Singapore, the Intermediary, and not Argus Research, is solely responsible for ensuring that the recipients of the Research Reports understand the information contained in the Research Reports and that such information is suitable based on the customer's profile and investment objectives."


### Commit changes with a computed column

Docling may emit `<!-- image -->` placeholders for figures. Headings
and body text follow those placeholders.

In [7]:
docs.add_computed_column(markdown=to_markdown(docs.document))

Added 1 column value with 0 errors in 2.97 s (0.34 rows/s)


1 row updated.

In [8]:
# Stored Markdown. Later inserts reuse this column definition.
docs.select(docs.title, docs.markdown).head(1)

title,markdown
Argus Market Digest,"<!-- image --> <!-- image --> <!-- image --> FRIDAY, JUNE 21, 2024 JUNE 20, DJIA: 39,134.76 UP 299.90 Good Morning. This is the Market Digest for Friday, June 21, 2024, with analysis of the financial markets and comments on Accenture plc. ## IN THIS ISSUE: * Growth Stock: Accenture plc: Shares rally on AI optimism (Jim Kelleher) ## MARKET REVIEW: Yogi Berra famously said ""When you come to a fork in the road, take it."" Stock investors did just that on Thursday, pushing the Dow Jones I ...... ures Act, the Intermediary accepts legal responsibility for the contents of Research Reports in respect of such recipient in accordance with applicable law. When reports are distributed by Intermediaries in Singapore, the Intermediary, and not Argus Research, is solely responsible for ensuring that the recipients of the Research Reports understand the information contained in the Research Reports and that such information is suitable based on the customer's profile and investment objectives."


### Store Markdown as a document

`document_splitter` reads a `pxt.Document` (a file), not a string. You
write the stored Markdown to a `.md` file so heading-aware chunking can
use Docling's structure.

In [9]:
@pxt.udf
def as_markdown_document(text: str) -> pxt.Document:
    import tempfile
    from pathlib import Path

    path = Path(tempfile.mkdtemp()) / 'converted.md'
    path.write_text(text, encoding='utf-8')
    # Pixeltable copies the file into document storage.
    return str(path)

In [10]:
docs.add_computed_column(markdown_doc=as_markdown_document(docs.markdown))

Added 1 column value with 0 errors in 0.01 s (76.44 rows/s)


1 row updated.

### Split into chunks

Create a view that splits on headings, then by token limit, so chunks
follow Docling's Markdown structure and stay within a model window.

In [11]:
chunks = pxt.create_view(
    'docling_demo/chunks',
    docs,
    iterator=document_splitter(
        docs.markdown_doc,
        separators='heading,token_limit',
        limit=300,
        metadata='heading',
    ),
)

Created view 'docling_demo/chunks'.


In [12]:
# Heading metadata comes from the Markdown Docling produced
chunks.select(chunks.heading, chunks.text).head(5)

heading,text
{},"FRIDAY, JUNE 21, 2024 JUNE 20, DJIA: 39,134.76 UP 299.90 Good Morning. This is the Market Digest for Friday, June 21, 2024, with analysis of the financial markets and comments on Accenture plc."
"{""h2"": ""IN THIS ISSUE:""}",IN THIS ISSUE: Growth Stock: Accenture plc: Shares rally on AI optimism (Jim Kelleher)
"{""h2"": ""MARKET REVIEW:""}","MARKET REVIEW: Yogi Berra famously said ""When you come to a fork in the road, take it."" Stock investors did just that on Thursday, pushing the Dow Jones Industrial Average higher by 0.77% but the Nasdaq Composite and S&P 500 lower by 0.79% and 0.25%, respectively. In a rare event, shares of Nvidia not only lost ground, but lost a relatively meaningful amount (3.5%), proving nothing can go up forever. Still, the major indices are comfortably ahead for the year to date - and the big non-AI mover for stocks is the future direction of interest rates, which remains a concern for Wall Street and (for one day at least) offset AI mania."
"{""h2"": ""ACCENTURE PLC (NYSE: ACN, \$306.16) BUY""}","ACCENTURE PLC (NYSE: ACN, \$306.16) BUY"
"{""h2"": ""ACN: Shares rally on AI optimism""}","ACN: Shares rally on AI optimism Accenture posted fiscal 3Q24 non-GAAP EPS that missed consensus forecasts. Both revenue and adjusted EPS edged lower year over year. Despite cautious fiscal 4Q24 guidance and reduced sales and non-GAAP EPS forecasts for FY24, the shares rallied on growing AI momentum for the business. Total bookings were strong in 3Q24, rising 22% annually on acceleration in managed services and resulting in a 1.3 book-to-bill. Accenture still appears to be taking share from competitors. We believe that Accenture has the financial resources, customer presence, and market strength to thrive as companies accelerate the process of digital transformation and begin their AI journeys."


### Add semantic search

Create an embedding index on the chunks. Embeddings run locally with a
sentence-transformers model. No API key.

In [13]:
chunks.add_embedding_index(
    column='text',
    string_embed=sentence_transformer.using(model_id='all-MiniLM-L6-v2'),
)

### Search converted documents

Use similarity search to find chunks by meaning, not keywords.

In [14]:
query = 'market trends'
sim = chunks.text.similarity(string=query)

results = (
    chunks.order_by(sim, asc=False)
    .select(chunks.text, score=sim)
    .limit(3)
)

In [15]:
# Closest chunks to the query
results.collect()

text,score
MARKET DIGEST,0.479
"MARKET REVIEW: Yogi Berra famously said ""When you come to a fork in the road, take it."" Stock investors did just that on Thursday, pushing the Dow Jones Industrial Average higher by 0.77% but the Nasdaq Composite and S&P 500 lower by 0.79% and 0.25%, respectively. In a rare event, shares of Nvidia not only lost ground, but lost a relatively meaningful amount (3.5%), proving nothing can go up forever. Still, the major indices are comfortably ahead for the year to date - and the big non-AI mover for stocks is the future direction of interest rates, which remains a concern for Wall Street and (for one day at least) offset AI mania.",0.448
"MARKET DIGEST In our view, Accenture has the size, scale, global reach, and broad industry expertise to continue growing faster than the industry and taking share from competitors. Accenture's share price, which rose in 2023 on competitive share gains and AI interest, has struggled in 2024 with the consulting market. From a valuation perspective, Accenture appears attractive based on our discounted free cash flow model and blended valuation analysis. We are reiterating our BUY rating and our 12-month target price of \$380.",0.426


### Convert only new documents

Insert another PDF. Pixeltable converts that row and builds its chunks
and embeddings. Existing Markdown is left unchanged.

In [16]:
docs.insert(
    [
        {
            'title': 'Company Research Alphabet',
            'document': (
                'https://raw.githubusercontent.com/'
                'pixeltable/pixeltable/main/docs/'
                'resources/rag-demo/'
                'Company-Research-Alphabet.pdf'
            ),
        }
    ]
)

Inserted 30 rows with 0 errors in 11.37 s (2.64 rows/s)


30 rows inserted.

In [17]:
# Two source documents. The first Markdown column is unchanged.
docs.select(docs.title).collect()


title
Argus Market Digest
Company Research Alphabet


## Explanation

**Computed columns cache converter output.** Inserting a row runs
`to_markdown()` for that row and stores the result. A later insert does
not re-run conversion on existing rows. Updating `document` invalidates
`markdown`, `markdown_doc`, the chunk view, and the embedding index.

**OCR is off in this recipe.** Digital PDFs do not need it. Set
`do_ocr=True` on `PdfPipelineOptions` when the source is a scan.

**This UDF stays in the notebook.** Docling is a converter you bring to
a computed column, not a built-in Pixeltable provider. For an
application file, move the UDF into a module and import it from
`app.py`.

## See also

- [Split documents into chunks for RAG](./doc-chunk-for-rag.ipynb)
- [Extract text from PowerPoint, Word, and Excel files](./doc-extract-text-from-office-files.ipynb)
- [UDFs in Pixeltable](/platform/udfs-in-pixeltable)